# ⚡ Energy Consumption Forecasting ML Pipeline (Google Colab)

This interactive Colab notebook runs the complete end-to-end time-series machine learning pipeline:
1. **Dataset Profiling & Intelligence** (`profiler.py`, `schema_detector.py`, `frequency_detector.py`, `validator.py`)
2. **Schema Normalization & Zero-Leakage Feature Engineering** (`normalizer.py`, `features.py`)
3. **Chronological Splitting & Candidate Model Training** (Naive Baseline, Random Forest, XGBoost, SARIMA)
4. **Objective Model Selection & Historical Retraining** (`evaluator.py`, `model_selector.py`)
5. **Genuine Recursive Multi-Step Future Forecasting** (`future_forecaster.py`)
6. **Interactive Visualizations & Model Artifact Downloads**

In [ ]:
# Step 1: Install Required Libraries in Colab
!pip install -q xgboost statsmodels joblib matplotlib seaborn

## Step 2: Load the Production Pipeline Code
Execute the cell below to load all dataset intelligence, preprocessing, modeling, evaluation, and multi-step recursive forecasting classes.

In [ ]:
import io
import json
import os
import re
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple, Union

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

# --- Data Structures & Exceptions ---
class SchemaDetectionError(Exception): pass
class AmbiguousSchemaError(SchemaDetectionError):
    def __init__(self, role: str, candidates: List[Tuple[str, float]]) -> None:
        self.role, self.candidates = role, candidates
        super().__init__(f"Ambiguous candidates for '{role}': {candidates}")

class PipelineValidationError(Exception):
    def __init__(self, errors: List[str]) -> None:
        super().__init__("Validation failed:\n - " + "\n - ".join(errors))

@dataclass
class ColumnCandidate:
    column_name: str
    confidence: float
    reasons: List[str] = field(default_factory=list)

@dataclass
class DetectedSchema:
    timestamp: str
    consumption: str
    temperature: Optional[str] = None
    humidity: Optional[str] = None
    wind_speed: Optional[str] = None
    pressure: Optional[str] = None
    holiday: Optional[str] = None
    def to_mapping(self) -> Dict[str, str]:
        m = {'timestamp': self.timestamp, 'consumption': self.consumption}
        for attr in ['temperature', 'humidity', 'wind_speed', 'pressure', 'holiday']:
            v = getattr(self, attr)
            if v: m[attr] = v
        return m

@dataclass
class FrequencyInfo:
    name: str; pandas_alias: str; median_delta_seconds: float
    periods_per_day: int; periods_per_week: int; is_regular: bool
    irregular_gap_percentage: float; recommended_lag_indices: Dict[str, int]
    resampling_recommended: bool

@dataclass
class EvaluationMetrics:
    model_name: str; mae: float; rmse: float; mape: float; wape: float; r2: float; sample_count: int
    def to_dict(self): return asdict(self)

# --- 1. Dataset Intelligence ---
class DatasetProfiler:
    def profile(self, df: pd.DataFrame) -> Dict[str, Any]:
        num_rows, num_cols = df.shape
        dup_c = int(df.duplicated().sum())
        tot_na = int(df.isna().sum().sum())
        return {
            'num_rows': num_rows, 'num_columns': num_cols,
            'column_names': list(df.columns),
            'duplicate_rows': dup_c, 'duplicate_row_percentage': round(dup_c/max(1, num_rows)*100, 2),
            'total_missing_cells': tot_na, 'missing_cell_percentage': round(tot_na/max(1, num_rows*num_cols)*100, 2),
        }

class SchemaDetector:
    TS_PATS = [(r'^timestamp$', 1.0), (r'^datetime$', 1.0), (r'^date_time$', 1.0), (r'^reading_time$', 0.95), (r'^date$', 0.85), (r'^time$', 0.85), (r'.*time.*', 0.65), (r'.*date.*', 0.65)]
    TGT_PATS = [(r'^consumption$', 1.0), (r'^energy_usage$', 1.0), (r'^power_consumption$', 1.0), (r'^electricity_demand$', 1.0), (r'^energy$', 0.95), (r'^load$', 0.90), (r'^demand$', 0.90), (r'^power$', 0.90), (r'^kwh$', 0.85), (r'^value$', 0.60)]
    WEA_PATS = {
        'temperature': [(r'^temperature$', 1.0), (r'^temp$', 0.95), (r'.*temp.*', 0.70)],
        'humidity': [(r'^humidity$', 1.0), (r'^hum$', 0.95), (r'.*hum.*', 0.70)],
    }
    def _match(self, name: str, patterns): 
        c = name.strip().lower().replace(' ', '_').replace('-', '_')
        return max([s if re.fullmatch(p, c) else (s*0.8 if re.search(p, c) else 0.0) for p, s in patterns] + [0.0])
    def detect_schema(self, df: pd.DataFrame, override_ts=None, override_tgt=None) -> DetectedSchema:
        # Timestamp detection
        ts_candidates = []
        for col in df.columns:
            series = df[col]
            n_score = self._match(str(col), self.TS_PATS)
            try:
                parse_r = float(pd.to_datetime(series.dropna().head(100).astype(str), errors='coerce').notna().mean())
            except Exception: parse_r = 0.0
            if parse_r >= 0.5 or n_score >= 0.5:
                conf = 0.6 * parse_r + 0.4 * n_score
                ts_candidates.append(ColumnCandidate(str(col), round(conf, 4)))
        ts_candidates.sort(key=lambda x: x.confidence, reverse=True)
        if not ts_candidates:
            raise SchemaDetectionError(f"No timestamp column found among {list(df.columns)}")
        sel_ts = override_ts or ts_candidates[0].column_name
        # Target detection
        tgt_candidates = []
        for col in df.columns:
            if str(col) == sel_ts: continue
            if pd.api.types.is_numeric_dtype(df[col]) or pd.to_numeric(df[col].dropna().head(50), errors='coerce').notna().mean() > 0.8:
                n_score = self._match(str(col), self.TGT_PATS)
                tgt_candidates.append(ColumnCandidate(str(col), round(0.7 * n_score + 0.3, 4)))
        tgt_candidates.sort(key=lambda x: x.confidence, reverse=True)
        if not tgt_candidates:
            rem = [c for c in df.columns if str(c) != sel_ts and pd.api.types.is_numeric_dtype(df[c])]
            if rem: sel_tgt = str(rem[0])
            else: raise SchemaDetectionError(f"No energy consumption column found among {list(df.columns)}")
        else:
            sel_tgt = override_tgt or tgt_candidates[0].column_name
        # Weather
        temp_col, hum_col = None, None
        for col in df.columns:
            if str(col) in [sel_ts, sel_tgt]: continue
            if self._match(str(col), self.WEA_PATS['temperature']) >= 0.7: temp_col = str(col)
            if self._match(str(col), self.WEA_PATS['humidity']) >= 0.7: hum_col = str(col)
        return DetectedSchema(timestamp=sel_ts, consumption=sel_tgt, temperature=temp_col, humidity=hum_col)

class FrequencyDetector:
    FREQS = [('15-minute', '15min', 900, 96, 672), ('30-minute', '30min', 1800, 48, 336), ('hourly', '1h', 3600, 24, 168), ('daily', '1D', 86400, 1, 7)]
    def detect_frequency(self, ts: pd.Series) -> FrequencyInfo:
        t = pd.to_datetime(ts, errors='coerce').dropna()
        delta_s = float(t.diff().dropna().dt.total_seconds().median()) if len(t) > 2 else 3600.0
        for name, alias, sec, ppd, ppw in self.FREQS:
            if abs(delta_s - sec) <= (sec * 0.15):
                if name == '15-minute': lags = {'lag_1': 1, 'lag_4': 4, 'lag_day': 96, 'lag_week': 672}
                elif name == '30-minute': lags = {'lag_1': 1, 'lag_2': 2, 'lag_day': 48, 'lag_week': 336}
                elif name == 'hourly': lags = {'lag_1': 1, 'lag_2': 2, 'lag_day': 24, 'lag_week': 168}
                else: lags = {'lag_1': 1, 'lag_2': 2, 'lag_week': 7, 'lag_month': 30}
                return FrequencyInfo(name, alias, delta_s, ppd, ppw, True, 0.0, lags, False)
        return FrequencyInfo('irregular', '1h', delta_s, 24, 168, False, 100.0, {'lag_1': 1, 'lag_day': 24}, True)

class DataValidator:
    def validate(self, df: pd.DataFrame) -> Tuple[bool, List[str], List[str]]:
        errors, warnings = [], []
        if 'timestamp' not in df.columns or 'consumption' not in df.columns: errors.append('Missing required columns.')
        if len(df) < 50: errors.append(f'Dataset has only {len(df)} rows (minimum 50 required).')
        if df['consumption'].std() < 1e-6: errors.append('Target series has zero variance (constant).')
        if (df['consumption'] < 0).any(): warnings.append('Contains negative consumption values (offset/generation).')
        return len(errors) == 0, errors, warnings

# --- 2. Preprocessing & Feature Engineering ---
class SchemaNormalizer:
    def normalize(self, df: pd.DataFrame, schema: DetectedSchema) -> pd.DataFrame:
        m = schema.to_mapping()
        rev_m = {v: k for k, v in m.items() if v in df.columns}
        norm = df[list(rev_m.keys())].rename(columns=rev_m)
        norm['timestamp'] = pd.to_datetime(norm['timestamp'], errors='coerce')
        norm = norm.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
        if norm['timestamp'].duplicated().any():
            norm = norm.groupby('timestamp', as_index=False).mean()
        for c in norm.columns:
            if c != 'timestamp':
                norm[c] = pd.to_numeric(norm[c], errors='coerce').astype(np.float64).ffill().bfill()
        return norm

class TimeSeriesFeatureEngineer:
    def __init__(self, freq_info: FrequencyInfo):
        self.freq_info, self.lags = freq_info, freq_info.recommended_lag_indices
        self.short_w = max(3, freq_info.periods_per_day // 4)
        self.long_w = max(4, freq_info.periods_per_day)
        self.warmup = max(max(self.lags.values()), self.long_w + 1)
        self.weather_cols, self.feature_cols = [], []
    def create_features(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
        f = df.copy()
        t = pd.to_datetime(f['timestamp'])
        f['hour'], f['day'], f['dayofweek'], f['month'] = t.dt.hour, t.dt.day, t.dt.dayofweek, t.dt.month
        f['hour_sin'] = np.sin(2 * np.pi * f['hour'] / 24.0)
        f['hour_cos'] = np.cos(2 * np.pi * f['hour'] / 24.0)
        f['weekday_sin'] = np.sin(2 * np.pi * f['dayofweek'] / 7.0)
        f['weekday_cos'] = np.cos(2 * np.pi * f['dayofweek'] / 7.0)
        for name, step in self.lags.items(): f[name] = f['consumption'].shift(step)
        # Strict shift(1) rolling to eliminate target leakage
        s_tgt = f['consumption'].shift(1)
        f[f'roll_mean_{self.short_w}'] = s_tgt.rolling(self.short_w, min_periods=1).mean()
        f[f'roll_std_{self.short_w}'] = s_tgt.rolling(self.short_w, min_periods=1).std().fillna(0.0)
        f[f'roll_mean_{self.long_w}'] = s_tgt.rolling(self.long_w, min_periods=1).mean()
        f[f'roll_min_{self.long_w}'] = s_tgt.rolling(self.long_w, min_periods=1).min()
        f[f'roll_max_{self.long_w}'] = s_tgt.rolling(self.long_w, min_periods=1).max()
        self.weather_cols = [c for c in ['temperature', 'humidity'] if c in f.columns]
        for w in self.weather_cols: f[f'{w}_lag_1'] = f[w].shift(1)
        self.feature_cols = [c for c in f.columns if c not in ['timestamp', 'consumption']]
        return f.iloc[self.warmup:].reset_index(drop=True), self.feature_cols
    def generate_single_step(self, hist_df: pd.DataFrame, next_ts: pd.Timestamp) -> pd.DataFrame:
        r = {'hour': next_ts.hour, 'day': next_ts.day, 'dayofweek': next_ts.dayofweek, 'month': next_ts.month,
             'hour_sin': np.sin(2*np.pi*next_ts.hour/24.0), 'hour_cos': np.cos(2*np.pi*next_ts.hour/24.0),
             'weekday_sin': np.sin(2*np.pi*next_ts.dayofweek/7.0), 'weekday_cos': np.cos(2*np.pi*next_ts.dayofweek/7.0)}
        cons = hist_df['consumption'].values
        for name, step in self.lags.items():
            r[name] = float(cons[-step]) if len(cons) >= step else float(cons[0])
        short_h = cons[-self.short_w:] if len(cons) >= self.short_w else cons
        long_h = cons[-self.long_w:] if len(cons) >= self.long_w else cons
        r[f'roll_mean_{self.short_w}'] = float(np.mean(short_h))
        r[f'roll_std_{self.short_w}'] = float(np.std(short_h)) if len(short_h) > 1 else 0.0
        r[f'roll_mean_{self.long_w}'] = float(np.mean(long_h))
        r[f'roll_min_{self.long_w}'] = float(np.min(long_h))
        r[f'roll_max_{self.long_w}'] = float(np.max(long_h))
        for w in self.weather_cols:
            r[w] = float(hist_df[w].iloc[-1]) if w in hist_df.columns else 0.0
            r[f'{w}_lag_1'] = r[w]
        return pd.DataFrame([r])[self.feature_cols]

# --- 3. Models & Evaluator ---
class BaselineForecaster:
    def __init__(self, seasonal_period=24): self.p = seasonal_period; self.last = 0.0
    def fit(self, X, y): self.last = float(y.iloc[-1]); return self
    def predict(self, X): return X['lag_1'].values if 'lag_1' in X.columns else np.full(len(X), self.last)

class RandomForestForecaster:
    def __init__(self): self.m = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
    def fit(self, X, y): self.cols = list(X.columns); self.m.fit(X.values, y.values); return self
    def predict(self, X): return self.m.predict(X[self.cols].values)

class XGBoostForecaster:
    def __init__(self):
        self.m = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, tree_method='hist') if HAS_XGBOOST else HistGradientBoostingRegressor(max_iter=100, max_depth=6, random_state=42)
    def fit(self, X, y): self.cols = list(X.columns); self.m.fit(X.values, y.values); return self
    def predict(self, X): return self.m.predict(X[self.cols].values)

class SARIMAForecaster:
    def __init__(self, s=24): self.s = min(s, 24); self.m = None; self.last = 0.0; self.fb = None
    def fit(self, X, y):
        y_arr = y.values; self.last = float(y_arr[-1])
        if HAS_STATSMODELS:
            try: self.m = SARIMAX(y_arr, order=(1,1,1), seasonal_order=(1,0,0,self.s), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=50)
            except Exception: self.fb = Ridge(alpha=1.0).fit(X.values, y_arr)
        else: self.fb = Ridge(alpha=1.0).fit(X.values, y_arr)
        return self
    def predict(self, X):
        if self.m: return np.nan_to_num(np.asarray(self.m.forecast(len(X))), nan=self.last)
        if self.fb: return self.fb.predict(X.values)
        return np.full(len(X), self.last)

class ModelEvaluator:
    def evaluate(self, y_true, y_pred, name='Model') -> EvaluationMetrics:
        yt, yp = np.asarray(y_true), np.asarray(y_pred)
        mae = float(np.mean(np.abs(yt - yp)))
        rmse = float(np.sqrt(np.mean(np.square(yt - yp))))
        mape = float(np.mean(np.abs(yt - yp) / np.maximum(np.abs(yt), 1e-5)) * 100.0)
        wape = float(np.sum(np.abs(yt - yp)) / max(np.sum(np.abs(yt)), 1e-5) * 100.0)
        ss_tot = float(np.sum(np.square(yt - np.mean(yt))))
        r2 = float(1.0 - (np.sum(np.square(yt - yp)) / max(ss_tot, 1e-5))) if ss_tot > 0 else 0.0
        return EvaluationMetrics(name, round(mae, 4), round(rmse, 4), round(mape, 2), round(wape, 2), round(r2, 4), len(yt))

# --- 4. Recursive Multi-Step Future Forecaster ---
class FutureForecaster:
    def generate_forecast(self, model, fe: TimeSeriesFeatureEngineer, hist_df: pd.DataFrame, freq: FrequencyInfo, horizon: int=24) -> pd.DataFrame:
        buf = hist_df.copy().reset_index(drop=True)
        buf['timestamp'] = pd.to_datetime(buf['timestamp'])
        curr_ts = buf['timestamp'].iloc[-1]
        delta = pd.to_timedelta(freq.median_delta_seconds, unit='s')
        points = []
        for step in range(1, horizon + 1):
            curr_ts = curr_ts + delta
            row_f = fe.generate_single_step(buf, curr_ts)
            pred_val = max(0.0, float(model.predict(row_f)[0]))
            points.append({'timestamp': curr_ts, 'forecast_consumption': round(pred_val, 4), 'step_ahead': step})
            new_r = {'timestamp': curr_ts, 'consumption': pred_val}
            for w in fe.weather_cols:
                new_r[w] = float(buf[w].iloc[-1]) if w in buf.columns else 0.0
            buf = pd.concat([buf, pd.DataFrame([new_r])], ignore_index=True)
        return pd.DataFrame(points)

# --- 5. Main Orchestrator ---
class EnergyForecastingPipeline:
    def __init__(self, models_dir='models'):
        self.models_dir = models_dir
        os.makedirs(models_dir, exist_ok=True)
        self.profiler = DatasetProfiler()
        self.detector = SchemaDetector()
        self.normalizer = SchemaNormalizer()
        self.validator = DataValidator()
        self.freq_det = FrequencyDetector()
        self.evaluator = ModelEvaluator()

    def run(self, data_source, horizon=24, model_tag='colab_model') -> Dict[str, Any]:
        raw_df = pd.read_csv(data_source) if isinstance(data_source, str) else data_source.copy()
        profile = self.profiler.profile(raw_df)
        schema = self.detector.detect_schema(raw_df)
        norm_df = self.normalizer.normalize(raw_df, schema)
        valid, errs, warns = self.validator.validate(norm_df)
        if not valid: raise PipelineValidationError(errs)
        freq = self.freq_det.detect_frequency(norm_df['timestamp'])
        fe = TimeSeriesFeatureEngineer(freq)
        feat_df, feat_cols = fe.create_features(norm_df)

        # Chronological 70% / 15% / 15% split
        n = len(feat_df)
        t_end, v_end = int(n * 0.70), int(n * 0.85)
        X_tr, y_tr = feat_df.iloc[:t_end][feat_cols], feat_df.iloc[:t_end]['consumption']
        X_va, y_va = feat_df.iloc[t_end:v_end][feat_cols], feat_df.iloc[t_end:v_end]['consumption']
        X_te, y_te = feat_df.iloc[v_end:][feat_cols], feat_df.iloc[v_end:]['consumption']

        candidates = {
            'Naive Baseline': BaselineForecaster(freq.periods_per_day),
            'Random Forest': RandomForestForecaster(),
            'XGBoost': XGBoostForecaster(),
            'SARIMA': SARIMAForecaster(freq.periods_per_day),
        }
        val_metrics = {}
        for name, m in candidates.items():
            try:
                m.fit(X_tr, y_tr)
                val_metrics[name] = self.evaluator.evaluate(y_va, m.predict(X_va), name)
            except Exception:
                val_metrics[name] = EvaluationMetrics(name, 9999, 9999, 999, 999, -1, len(y_va))

        # Model Selection by Validation MAE
        best_name = min(val_metrics.keys(), key=lambda k: val_metrics[k].mae)

        # Retrain on full historical data (Train + Validation)
        X_tv = pd.concat([X_tr, X_va], axis=0).reset_index(drop=True)
        y_tv = pd.concat([y_tr, y_va], axis=0).reset_index(drop=True)
        best_model = candidates[best_name]
        best_model.fit(X_tv, y_tv)

        # Out-of-sample Test evaluation
        test_preds = best_model.predict(X_te)
        test_metric = self.evaluator.evaluate(y_te, test_preds, best_name)

        # Recursive Future Forecast
        ff = FutureForecaster()
        forecast_df = ff.generate_forecast(best_model, fe, norm_df, freq, horizon=horizon)

        # Persistence
        model_p = os.path.join(self.models_dir, f'{model_tag}_{best_name.lower().replace(" ", "_")}.joblib')
        meta_p = os.path.join(self.models_dir, f'{model_tag}_metadata.json')
        joblib.dump(best_model, model_p)
        with open(meta_p, 'w') as f: json.dump({'selected_model': best_name, 'test_metrics': test_metric.to_dict()}, f, indent=2)

        return {
            'profile': profile, 'schema': schema.to_mapping(), 'frequency': freq.name,
            'val_metrics': val_metrics, 'best_model': best_name, 'test_metrics': test_metric,
            'forecast_df': forecast_df, 'model_path': model_p, 'metadata_path': meta_p,
            'test_actual': y_te.values, 'test_preds': test_preds, 'test_timestamps': feat_df.iloc[v_end:]['timestamp'].values,
            'norm_df': norm_df
        }

print("✅ Pipeline classes and modules successfully initialized!")

## Step 3: Choose Data Input (Upload CSV or Generate Synthetic Data)
Run the cell below to upload your own CSV file, or generate a realistic 30-day hourly synthetic energy dataset.

In [ ]:
from google.colab import files

print("Options:")
print("1. Click 'Cancel upload' to use built-in synthetic realistic energy data.")
print("2. Choose your own energy consumption CSV file.")

uploaded = files.upload()
if uploaded:
    filename = list(uploaded.keys())[0]
    df_input = pd.read_csv(io.BytesIO(uploaded[filename]))
    print(f"✅ Successfully loaded uploaded file: {filename} with shape {df_input.shape}")
else:
    print("Generating synthetic 30-day hourly dataset with weather & diurnal cycle...")
    np.random.seed(42)
    dates = pd.date_range('2026-01-01', periods=720, freq='1h')
    hour = dates.hour.to_numpy()
    temp = 18.0 + 8.0 * np.sin(2 * np.pi * (hour - 9) / 24) + np.random.normal(0, 1.0, 720)
    hum = 60.0 - 20.0 * np.sin(2 * np.pi * (hour - 9) / 24) + np.random.normal(0, 2.5, 720)
    power = 35.0 + 15.0 * np.sin(2 * np.pi * (hour - 6) / 24) + 0.9 * np.maximum(0, temp - 22.0) + np.random.normal(0, 2.0, 720)
    df_input = pd.DataFrame({
        'DateTime': dates.strftime('%Y-%m-%d %H:%M:%S'),
        'Energy_Usage': np.round(power, 2),
        'Temperature': np.round(temp, 1),
        'Humidity': np.round(hum, 1)
    })
    print(f"✅ Generated synthetic dataset with {len(df_input)} rows!")

df_input.head()

## Step 4: Execute the Complete ML Training & Forecasting Pipeline
Runs dataset intelligence, schema detection, validation, feature engineering, candidate model training, selection, history retraining, and recursive 24-step future forecasting.

In [ ]:
pipeline = EnergyForecastingPipeline(models_dir='models')
results = pipeline.run(df_input, horizon=24, model_tag='colab_energy')

print("=" * 70)
print("  PIPELINE EXECUTION SUMMARY")
print("=" * 70)
print(f"Detected Schema: {results['schema']}")
print(f"Detected Frequency: {results['frequency']}")
print(f"Selected Winner: >>> {results['best_model']} <<<")

print("\n--- Candidate Model Validation Leaderboard ---")
val_table = pd.DataFrame([{
    'Model': k, 'MAE': v.mae, 'RMSE': v.rmse, 'MAPE (%)': f'{v.mape:.2f}%', 'WAPE (%)': f'{v.wape:.2f}%', 'R2': v.r2
} for k, v in results['val_metrics'].items()]).sort_values('MAE').reset_index(drop=True)
display(val_table)

print("\n--- Held-Out Test Set Performance ---")
tm = results['test_metrics']
print(f"  MAE:  {tm.mae:.4f} | RMSE: {tm.rmse:.4f} | MAPE: {tm.mape:.2f}% | R2: {tm.r2:.4f}")

print("\n--- Recursive Multi-Step Future Forecast (Horizon=24) ---")
display(results['forecast_df'].head(10))

## Step 5: Visualizations & Forecast Plot
Visualize historical test predictions against actuals, followed by the true multi-step future forecast curve.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Test Set - Actual vs Predicted
test_ts = pd.to_datetime(results['test_timestamps'])
ax1.plot(test_ts, results['test_actual'], label='Actual Consumption', color='#1f77b4', lw=2)
ax1.plot(test_ts, results['test_preds'], label=f"{results['best_model']} (Test Predictions)", color='#ff7f0e', lw=2, linestyle='--')
ax1.set_title(f"Held-Out Test Set Evaluation ({results['best_model']}) - MAE: {tm.mae:.2f}", fontsize=13, fontweight='bold')
ax1.set_ylabel('Consumption', fontsize=11)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: Recent History + Genuine Future Forecast
norm_df = results['norm_df']
recent_hist = norm_df.tail(72)
ax2.plot(recent_hist['timestamp'], recent_hist['consumption'], label='Recent Actual History', color='#2ca02c', lw=2)
ax2.plot(results['forecast_df']['timestamp'], results['forecast_df']['forecast_consumption'], label='Multi-Step Recursive Future Forecast', color='#d62728', lw=2.5, marker='o', markersize=4, linestyle='--')
ax2.axvline(x=recent_hist['timestamp'].iloc[-1], color='black', linestyle=':', label='Forecast Origin (t_0)')
ax2.set_title(f"Recursive Multi-Step Future Forecast (Horizon=24 Steps)", fontsize=13, fontweight='bold')
ax2.set_xlabel('Timestamp', fontsize=11)
ax2.set_ylabel('Consumption', fontsize=11)
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Download Saved Model Artifacts (.joblib & .json)
Download the trained model and metadata configuration to your local machine for deployment.

In [ ]:
from google.colab import files

print(f"Downloading model artifact: {results['model_path']}")
files.download(results['model_path'])

print(f"Downloading metadata artifact: {results['metadata_path']}")
files.download(results['metadata_path'])